---
**Non-Parametric & Proportion Tests in Python**
Data Analysis Course · Chapter 8
---

This notebook is the Python equivalent of the R Markdown `_07_hypothesis_testing_prop.Rmd`.
Topics: **non-parametric tests** (Wilcoxon/Mann-Whitney), **proportion tests** (Fisher's exact test,
chi-square), and the **power of a test**.

Work through it cell by cell — run each code cell with **Shift+Enter**.

**Required packages:** `numpy`, `pandas`, `matplotlib`, `scipy`, `pyreadr`
```
pip install numpy pandas matplotlib scipy pyreadr
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import pyreadr, urllib.request

We work with tumor expression data of AML/ALL leukemia patients:

In [ ]:
all_aml = pd.read_csv(
    "https://www.dropbox.com/scl/fi/q4rbpl3un6oqbbwoxe1qw/all.aml.cleaned.csv?rlkey=igwbna22d6hpdmw3os79zoy7z&dl=1",
    sep="\t"
)
all_aml = all_aml.set_index(all_aml.columns[0])   # R: data.matrix(all.aml), genes as rows

urllib.request.urlretrieve(
    "https://www.dropbox.com/scl/fi/0lqjh3jcskki2bb8hxwj6/all.aml.anno.rds?rlkey=cijsx39kr9gx9bjl6wzpmkgnz&dl=1",
    "all_aml_anno.rds"
)
all_aml_anno = pyreadr.read_r("all_aml_anno.rds")[None]

i_all = all_aml_anno.index[all_aml_anno["ALL.AML"] == "AML"]   # note: same naming quirk as the R original

## 0 – Introduction and objectives

Last time we covered hypothesis testing and the t-test. This week: **non-parametric tests** and
**proportion testing**.

## 1 – Non-parametric test

What if the data is not normally distributed? Then a t-test isn't valid. Let's check the
distribution of gene FOSB:

In [ ]:
expression = all_aml.loc["FOSB"]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(expression, bins=30)
stats.probplot(expression, dist="norm", plot=axes[1])
plt.show()

# Everything but normal!

Since the data isn't normal, we can't use the t-test — we need a **non-parametric test**: the
**Wilcoxon rank-sum test** (a.k.a. Mann-Whitney U test), `scipy.stats.mannwhitneyu()`. Like
Spearman vs. Pearson correlation, this test works on **ranks**, not raw values.

In [ ]:
exp_all = all_aml.loc["FOSB", i_all]
exp_aml = all_aml.loc["FOSB"].drop(index=i_all)

# R: wilcox.test(exp.all, exp.aml)
result_wilcox = stats.mannwhitneyu(exp_all, exp_aml)
print(result_wilcox)

In [ ]:
# Compare with the (invalid, here) t-test p-value:
result_ttest = stats.ttest_ind(exp_all, exp_aml)
print(result_ttest)

# The p-values are very different! We CANNOT trust the t-test here due to non-normality —
# the correct p-value is the one from the Wilcoxon test.
# Try another random gene!

## 2 – Proportion tests

The t-test and Wilcoxon test are **tests of the mean**. But sometimes we want to test the
relationship between two **categorical** variables — using a **contingency table** and Fisher's
exact test or the chi-square test.

In [ ]:
dat_brca = pd.read_csv(
    "https://www.dropbox.com/scl/fi/gy2jnj7o7p1398azkw6b1/gbsg_ba_ca.dat?rlkey=a0j003a1ehtt68usuyd32ta56&dl=1",
    sep="\t"
)

Is the choice of tamoxifen treatment (`hormon`) related to menopausal status (`meno`)? First, the contingency table:

In [ ]:
# R: table(dat.brca$meno, dat.brca$hormon)
ct = pd.crosstab(dat_brca["meno"], dat_brca["hormon"])
ct

The **odds-ratio (OR)**:

In [ ]:
OR = (ct.iloc[0, 0] / ct.iloc[0, 1]) / (ct.iloc[1, 0] / ct.iloc[1, 1])
OR

# How would the odds-ratio look if you transposed the table?

Now the one-sided Fisher Exact Test:

> H0: the odds-ratio is not significantly larger than one
> H1: the odds-ratio is significantly larger than one

In [ ]:
# R: fisher.test(tab, alternative='greater')
odds_ratio, p_value = stats.fisher_exact(ct, alternative="greater")
print("odds ratio:", odds_ratio, " p-value:", p_value)

# Check that this odds-ratio matches your manual computation above!
# Now run the two-sided Fisher test — first formulate the hypothesis!
# Check the p-values: what do you observe?

We can also use the **chi-square test**, comparing observed vs. expected counts under
independence:

> H0: observed and expected occurrences are not significantly different
> H1: observed and expected occurrences are significantly different

In [ ]:
# R: chisq.test(tab)
chi2, p, dof, expected = stats.chi2_contingency(ct)
print(f"chi2={chi2}, p-value={p}, dof={dof}")

# Is this a one-sided or two-sided test?

Now: does age (over/under 40) affect tumor grade?

In [ ]:
# R: table(dat.brca$age>40, dat.brca$grade)
tab = pd.crosstab(dat_brca["age"] > 40, dat_brca["grade"])
tab

In [ ]:
chi2, p, dof, expected = stats.chi2_contingency(tab)
print(f"chi2={chi2}, p-value={p}, dof={dof}")

In [ ]:
# R: tot = apply(tab, 2, sum)  — column totals
tot = tab.sum(axis=0)
print(tot)

# R: age = apply(tab, 1, sum)  — row totals
age_totals = tab.sum(axis=1)
print(age_totals)

In [ ]:
tot_proportions = tot / tot.sum()

# R: tab.exp = sapply(tot.proportions, function(x) x * age)
tab_exp = pd.DataFrame(
    np.outer(age_totals, tot_proportions),
    index=tab.index, columns=tab.columns
)
tab_exp

# How would you compute the chi-square test statistic manually from `tab` and `tab_exp`?
# (Hint: this is exactly what `expected` from chi2_contingency() above already gives you.)

---
## Exercises

### Exercise 1

1. Check the expression of a random gene in the ALL/AML dataset. Are the values normally distributed?
2. Run both a t-test and a Wilcoxon test on this gene. Is there a difference between the p-values?
   What did you expect?
3. Run both tests on **every** gene and store each p-value.
4. Scatter-plot -log10(p-value of t-test) vs. -log10(p-value of Wilcoxon test). Are there strong deviations?

In [ ]:
# Your code here:

### Exercise 2

What test would you use for the following questions?

- A lotion company wants to know whether their product is more likely to cause acne in men than in women
- The education department wants to know whether social science students have higher grades than science students
- A biologist wants to know whether a specific gene is more likely to be silenced in lactose-intolerant people

In [ ]:
# Your code here (as text/markdown reasoning is fine):

### Exercise 3

A pharmaceutical company's new vaccine works on 178 out of 200 patients. 75% of the batch had been
previously vaccinated with another vaccine, and it worked on 143 of those. Is there a significant
disproportion suggesting higher efficacy in previously vaccinated people, at alpha=0.05?
*(Hint: build a 2×2 contingency table and use `stats.fisher_exact` or `stats.chi2_contingency`.)*

In [ ]:
# Your code here:

## Summary: What have we learned?

| R | Python | Purpose |
|---|--------|---------|
| `wilcox.test(x, y)` | `scipy.stats.mannwhitneyu(x, y)` | Wilcoxon / Mann-Whitney non-parametric test |
| `table(x, y)` | `pd.crosstab(x, y)` | Contingency table |
| `fisher.test(tab, alternative=)` | `scipy.stats.fisher_exact(tab, alternative=)` | Fisher's exact test (2×2 tables) |
| `chisq.test(tab)` | `scipy.stats.chi2_contingency(tab)` | Chi-square test of independence |
| `apply(tab, 2, sum)` / `apply(tab, 1, sum)` | `tab.sum(axis=0)` / `tab.sum(axis=1)` | Row/column totals |